***Stochastic Gradient Descent (SGD) From Scratch***

**1. Introduction**

Stochastic Gradient Descent (SGD) is an optimization algorithm used to minimize a loss function by updating model parameters using one randomly selected training sample at a time.

In this notebook, I implement a custom Stochastic Gradient Descent algorithm from scratch using NumPy and apply it to a diabetes regression dataset.

**2. Objectives**

Understand how Stochastic Gradient Descent works.

Implement SGD from scratch using NumPy.

Apply feature scaling using StandardScaler.

Train a regression model using randomly shuffled training samples.

Predict target values for unseen test data.

Evaluate model performance using the coefficient of determination ($(R^2)$).

In [460]:
import numpy as np
import pandas as pd


**3. Dataset Description**

The dataset is loaded from Diabetes.csv.

Features (X): All columns except Target.

Target (y): The Target column, representing the continuous variable to be predicted.

The dataset is divided into training and testing sets using an 80:20 ratio.



In [461]:
df=pd.read_csv(r"C:\Users\Niloy\Downloads\ml practice\ML Algorithms\Diabetes.csv")

In [462]:
df

,BMI,BP,Cholesterol,LDL,Target
0,26,99,165,75,524
1,39,65,285,92,717
2,34,70,226,77,655
3,31,114,274,63,732
4,21,96,198,68,606
...,...,...,...,...,...
195,26,116,221,181,784
196,34,112,250,190,902
197,38,86,176,135,703
198,38,105,179,154,747


In [463]:
X=df.drop("Target",axis=1)
y=df["Target"]

**4. Data Preprocessing**

Train-Test Split

The dataset is split using train_test_split() with test_size=0.2 and random_state=42.

Feature Scaling

StandardScaler is used to standardize the input features.

The scaler is fitted only on the training data to prevent data leakage. The same transformation is then applied to the test data.

In [464]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [465]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**5. SGD Algorithm Implementation**

A custom SGD class is created with the following methods:

__init__(): Initializes the learning rate, number of epochs, weights, and bias.

fit(): Trains the model by updating the weights and bias using one randomly selected sample at a time.

predict(): Predicts target values using the learned parameters.

co_efficient(): Returns the learned weights.

intercept(): Returns the learned bias.

Prediction Equation

For a dataset with multiple input features, the prediction is calculated as

$$
\hat{y}i = \sum{j=1}^{n} w_j x_{ij} + b
$$

where:

$\hat{y}_i$ = predicted target value

$x_{ij}$ = value of feature (j) for sample (i)

$w_j$ = weight associated with feature (j)

(b) = bias or intercept

Error Calculation

$$
e_i = \hat{y}_i-y_i
$$

Parameter Updates

For the mean squared error objective, using the gradient without the constant factor of 2:

$$
w_j \leftarrow w_j-\eta e_i x_{ij}
$$

$$
b \leftarrow b-\eta e_i
$$

where $\eta$ is the learning rate.

At the beginning of each epoch, the training samples are randomly shuffled using np.random.permutation().

In [466]:
class SGD:
    def __init__(self,learning_rate,epoch):
        self.lr=learning_rate
        self.epo = epoch
        self.weight = None
        self.bias = None
    def fit(self,X_train,y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float).reshape(-1)

        self.weight = np.zeros(X_train.shape[1])
        self.bias = 0
        for epoch in range(self.epo):
            indices=np.random.permutation(X_train.shape[0])
            for i in indices:
                y_pred = np.dot(X_train[i],self.weight)+self.bias
                error=y_pred-y_train[i]

                dw=error*X_train[i]
                db=error

                self.weight-=self.lr*dw
                self.bias-=self.lr*db
    def predict(self,X_test):
        X_test=np.asarray(X_test, dtype=float)
        return np.dot(X_test,self.weight)+self.bias
    def co_efficient(self):
        return self.weight
    def intercept(self):
        return self.bias
    


**6. Model Training**

The model is trained using:

Learning rate: 0.01

Number of epochs: 100

Initial weights: Zeros

Initial bias: Zero

During each epoch, the model processes every training sample individually and updates its parameters after each sample.

In [467]:
sgd=SGD(learning_rate=0.01,epoch=100)

In [468]:
sgd.fit(X_train_scaled,y_train)

**7. Model Prediction**

After training, the model predicts the target values for the scaled test dataset.

These predictions are compared with the actual target values to evaluate the model's performance.

In [469]:
y_pred=sgd.predict(X_test_scaled)
y_pred

array([702.97404383, 761.09746246, 716.07369811, 866.06699501,
       503.77748844, 758.7038026 , 741.7598173 , 580.86638034,
       819.30826988, 927.4671684 , 588.13059452, 666.67150625,
       741.86257906, 914.82916032, 644.61217392, 794.16097323,
       563.00515374, 734.22335053, 586.24114502, 533.72393852,
       799.75362554, 523.27685938, 773.51553594, 565.89604079,
       605.36035135, 594.28275092, 822.75029466, 711.23276793,
       794.32096656, 777.17681792, 662.27573721, 820.13762871,
       627.31317875, 928.71383431, 626.74380272, 661.18887561,
       835.78540779, 759.53562855, 559.7151882 , 557.25074723])

**8. Model Evaluation**

The coefficient of determination ((R^2)) is used to evaluate the regression model.

$$
R^2=1-\frac{\sum_{i=1}^{m}(y_i-\hat{y}i)^2}
{\sum{i=1}^{m}(y_i-\bar{y})^2}
$$

where:

$y_i$ = actual target value

$\hat{y}_i$ = predicted target value

$\bar{y}$ = mean of actual target values

(m) = number of test samples

An $R^2$ value closer to 1 generally indicates better predictive performance relative to predicting the test-set mean.

In [470]:
from sklearn.metrics import r2_score
r2=r2_score(y_test,y_pred)
print("R^2 score",r2)

R^2 score 0.9448227676048089
